# Chapter 2.1 - Your first LangChain chain

**Goal:** understand what a *chain* is by doing the same task two ways:
1. **Manually** - call prompt, LLM and parser one after another.
2. **As a chain** - pipe the same pieces together with `|` (LCEL, LangChain Expression Language).

A chain here = `Prompt  ->  LLM  ->  Output parser`.

In [ ]:
# Chat model client. ChatOpenAI works with any OpenAI-compatible API,
# here we point it at NVIDIA's endpoint via base_url.
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # API key is read automatically from the OPENAI_API_KEY env var
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b"
)

In [2]:
# Message classes (not used below, kept for reference) and the prompt template class
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate


## First Chain
Build the 3 building blocks (prompt, LLM, parser) separately, then use them manually and as a chain.

In [4]:
#Task-1[Prompt]
# A template with a system message and a user message.
# {input} is a placeholder that gets filled in at invoke() time.
prompt_template = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant"),
    ("user","{input}")
])

In [3]:
#Task-2[LLM]
# Same LLM as the first cell (redefined here so this section is self-contained)
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

In [6]:
#Task-3[Str parser]
# The LLM returns an AIMessage object; StrOutputParser extracts just the text (.content)
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()


## Manual Execution
Run each step by hand: **prompt -> LLM -> take `.content`**. This is exactly what a chain automates.

In [8]:
# Step 1: fill the template -> gives a prompt value (system + user messages)
template = prompt_template.invoke({"input":"What is the capital of France?"})

# Step 2: send it to the LLM -> returns an AIMessage
result= llm.invoke(template)

# Step 3: pull the plain text out of the AIMessage (what StrOutputParser does for us)
final_result=result.content

print(final_result)

The capital of France is Paris.


## Chain Invocation
Same thing, but the `|` operator links the steps. The output of each step becomes the input of the next.

In [9]:
# `|` builds a RunnableSequence: prompt -> llm -> parser
chain = prompt_template | llm | str_parser

# Input is a dict because the prompt needs the key "input"
chain.invoke({"input":"What is the capital of France?"})

'The capital of France is Paris.'

In [10]:
# Same chain written explicitly with RunnableSequence (what `|` does under the hood)
from langchain_core.runnables import RunnableSequence

chain_1 = RunnableSequence(prompt_template,llm,str_parser)

# A plain string works here because the prompt has only ONE variable ({input}),
# so LangChain maps the string to it automatically
chain_1.invoke("What is the capital of France?")

'The capital of France is Paris.'